<a href="https://colab.research.google.com/github/AsimobiChisomSandra/ATF_Training_Practice/blob/main/Week3_12_guardrails_safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.3: Guardrails - Safe AI Systems

## Key Insight: How to mitigate undesired outputs from LLMs

Make your digital twin SAFE for production!

## What You'll Build
- Content filters
- Output validators
- Safety layers
- Fallback strategies
- PII detection

In [1]:
import re
from typing import List, Dict, Any, Optional

print('Week 3.3: Guardrails & Safety')
print('Goal: Make your twin safe for real users!')

Week 3.3: Guardrails & Safety
Goal: Make your twin safe for real users!


## Part 1: Content Filtering

Block inappropriate content.

In [2]:
class ContentFilter:
    '''Filter inappropriate content.'''

    def __init__(self):
        # Simplified word lists for demo
        self.profanity_list = ['badword1', 'badword2']  # Add actual words
        self.sensitive_topics = ['violence', 'illegal']

    def check_profanity(self, text: str) -> Dict[str, Any]:
        '''Check for profanity.'''
        text_lower = text.lower()
        found = [word for word in self.profanity_list if word in text_lower]

        return {
            'is_safe': len(found) == 0,
            'violations': found
        }

    def check_sensitive_topics(self, text: str) -> Dict[str, Any]:
        '''Check for sensitive topics.'''
        text_lower = text.lower()
        found = [topic for topic in self.sensitive_topics if topic in text_lower]

        return {
            'is_safe': len(found) == 0,
            'topics': found
        }

    def filter(self, text: str) -> Dict[str, Any]:
        '''Run all filters.'''
        profanity_check = self.check_profanity(text)
        topic_check = self.check_sensitive_topics(text)

        is_safe = profanity_check['is_safe'] and topic_check['is_safe']

        return {
            'is_safe': is_safe,
            'profanity_violations': profanity_check['violations'],
            'sensitive_topics': topic_check['topics'],
            'text': text if is_safe else '[FILTERED]'
        }

filter_system = ContentFilter()

print('Content Filter Tests:')
print(filter_system.filter('Hello, how are you?'))
print(filter_system.filter('This involves violence'))  # Will be flagged

Content Filter Tests:
{'is_safe': True, 'profanity_violations': [], 'sensitive_topics': [], 'text': 'Hello, how are you?'}
{'is_safe': False, 'profanity_violations': [], 'sensitive_topics': ['violence'], 'text': '[FILTERED]'}


## Part 2: PII Detection

Detect and protect personal information.

In [3]:
class PIIDetector:
    '''Detect personally identifiable information.'''

    def __init__(self):
        # Regex patterns for common PII
        self.patterns = {
            'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'phone': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'credit_card': r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b'
        }

    def detect(self, text: str) -> Dict[str, Any]:
        '''Detect PII in text.'''
        found_pii = {}

        for pii_type, pattern in self.patterns.items():
            matches = re.findall(pattern, text)
            if matches:
                found_pii[pii_type] = matches

        return {
            'has_pii': len(found_pii) > 0,
            'pii_types': list(found_pii.keys()),
            'details': found_pii
        }

    def redact(self, text: str) -> str:
        '''Redact PII from text.'''
        result = text

        for pii_type, pattern in self.patterns.items():
            result = re.sub(pattern, f'[REDACTED_{pii_type.upper()}]', result)

        return result

pii_detector = PIIDetector()

text_with_pii = "Contact me at john@example.com or call 555-123-4567"
print('\nPII Detection:')
print('Original:', text_with_pii)
print('Detection:', pii_detector.detect(text_with_pii))
print('Redacted:', pii_detector.redact(text_with_pii))


PII Detection:
Original: Contact me at john@example.com or call 555-123-4567
Detection: {'has_pii': True, 'pii_types': ['email', 'phone'], 'details': {'email': ['john@example.com'], 'phone': ['555-123-4567']}}
Redacted: Contact me at [REDACTED_EMAIL] or call [REDACTED_PHONE]


## Part 3: Output Validator

Ensure AI outputs meet requirements.

In [4]:
class OutputValidator:
    '''Validate AI outputs before showing to user.'''

    def __init__(self, max_length: int = 500):
        self.max_length = max_length
        self.content_filter = ContentFilter()
        self.pii_detector = PIIDetector()

    def validate(self, output: str) -> Dict[str, Any]:
        '''Run all validation checks.'''
        issues = []

        # Check length
        if len(output) > self.max_length:
            issues.append(f'Too long ({len(output)} > {self.max_length} chars)')

        # Check if empty
        if not output.strip():
            issues.append('Empty output')

        # Check content safety
        content_check = self.content_filter.filter(output)
        if not content_check['is_safe']:
            issues.append('Content safety violation')

        # Check for PII
        pii_check = self.pii_detector.detect(output)
        if pii_check['has_pii']:
            issues.append(f"Contains PII: {pii_check['pii_types']}")

        # Check for disclaimers (if needed)
        needs_disclaimer = any(word in output.lower()
                             for word in ['medical', 'legal', 'financial'])

        return {
            'is_valid': len(issues) == 0,
            'issues': issues,
            'needs_disclaimer': needs_disclaimer,
            'output': output
        }

    def add_disclaimer(self, output: str, topic: str) -> str:
        '''Add disclaimer to output.'''
        disclaimers = {
            'medical': "\n\n⚠️ Note: This is not medical advice. Consult a healthcare professional.",
            'legal': "\n\n⚠️ Note: This is not legal advice. Consult a lawyer.",
            'financial': "\n\n⚠️ Note: This is not financial advice. Consult a financial advisor."
        }

        for key, disclaimer in disclaimers.items():
            if key in topic.lower():
                return output + disclaimer

        return output

validator = OutputValidator(max_length=300)

test_output = "You should invest all your money in stocks."
result = validator.validate(test_output)

print('\nValidation Result:')
print(f"Valid: {result['is_valid']}")
print(f"Issues: {result['issues']}")
print(f"Needs disclaimer: {result['needs_disclaimer']}")

if result['needs_disclaimer']:
    safe_output = validator.add_disclaimer(test_output, 'financial')
    print(f"\nWith disclaimer: {safe_output}")


Validation Result:
Valid: True
Issues: []
Needs disclaimer: False


## Part 4: Safe Digital Twin

Put it all together.

In [5]:
class SafeDigitalTwin:
    '''Digital twin with safety guardrails.'''

    def __init__(self, name: str):
        self.name = name
        self.validator = OutputValidator()
        self.content_filter = ContentFilter()

    def _process_input(self, user_input: str) -> Optional[str]:
        '''Process and filter user input.'''
        # Filter input
        filter_result = self.content_filter.filter(user_input)

        if not filter_result['is_safe']:
            return None  # Input blocked

        return filter_result['text']

    def _generate_response(self, filtered_input: str) -> str:
        '''Generate AI response (mock for demo).'''
        # This would call your actual AI
        return f"I received: {filtered_input}"

    def _process_output(self, response: str) -> str:
        '''Validate and process output.'''
        validation = self.validator.validate(response)

        if not validation['is_valid']:
            # Fallback to safe response
            return "I apologize, but I cannot provide that response. How else can I help?"

        # Add disclaimers if needed
        if validation['needs_disclaimer']:
            response = self.validator.add_disclaimer(response, response)

        return response

    def chat(self, user_input: str) -> str:
        '''Safe chat with full pipeline.'''
        # Step 1: Filter input
        filtered_input = self._process_input(user_input)

        if filtered_input is None:
            return "I cannot process that input. Please rephrase."

        # Step 2: Generate response
        response = self._generate_response(filtered_input)

        # Step 3: Validate and process output
        safe_response = self._process_output(response)

        return safe_response

safe_twin = SafeDigitalTwin('Safe Assistant')

print('\nSafe Digital Twin:')
print(safe_twin.chat('Hello, how are you?'))
print(safe_twin.chat('Tell me about medical treatments'))  # Will add disclaimer


Safe Digital Twin:
I received: Hello, how are you?
I received: Tell me about medical treatments

⚠️ Note: This is not medical advice. Consult a healthcare professional.


## Exercise 1: Add More Guardrails

**Task**: Enhance the safety system:

1. Add more PII patterns (addresses, dates of birth)
2. Create custom content filters for your domain
3. Add rate limiting
4. Implement logging of safety violations
5. Test with edge cases

In [7]:
import time
import re
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional

# --- 1. Enhanced PII Detector ---
class EnhancedPIIDetector(PIIDetector):
    def __init__(self):
        super().__init__()
        # Adding patterns for Date of Birth (DOB) and basic US Physical Addresses
        self.patterns['dob'] = r'\b(?:(?:0[1-9]|1[0-2])[-/./ ]?(?:0[1-9]|[12]\d|3[01])[-/./ ]?(?:19|20)\d{2})\b'
        self.patterns['address'] = r'\b\d{1,6}\s+(?:[A-Za-z0-9#./\s]{2,30})\s+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Drive|Dr|Lane|Ln|Way|Court|Ct)\b'

# --- 2. Custom Domain Filters ---
class DomainFilter(ContentFilter):
    def __init__(self):
        super().__init__()
        # Add custom business domain restrictions
        self.restricted_keywords = ['competitor_x', 'insider_info', 'leak_secrets']

    def check_business_compliance(self, text: str) -> Dict[str, Any]:
        text_lower = text.lower()
        found = [kw for kw in self.restricted_keywords if kw in text_lower]
        return {
            'is_safe': len(found) == 0,
            'violations': found
        }

    def filter(self, text: str) -> Dict[str, Any]:
        # Combine base filtering with business compliance
        base_result = super().filter(text)
        compliance_check = self.check_business_compliance(text)

        is_safe = base_result['is_safe'] and compliance_check['is_safe']
        violations = base_result['sensitive_topics'] + compliance_check['violations']

        return {
            'is_safe': is_safe,
            'profanity_violations': base_result['profanity_violations'],
            'sensitive_topics': violations,
            'text': text if is_safe else '[FILTERED_DOMAIN_VIOLATION]'
        }

# --- 3. Rate Limiting Guardrail ---
class RateLimiter:
    def __init__(self, max_requests: int = 3, window_seconds: int = 10):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests = []

    def is_allowed(self) -> bool:
        current_time = time.time()
        # Keep only requests within the active window
        self.requests = [t for t in self.requests if current_time - t < self.window_seconds]

        if len(self.requests) < self.max_requests:
            self.requests.append(current_time)
            return True
        return False

# --- 4. Safety Violation Logging System ---
safety_audit_logs = []

def log_security_event(event_type: str, details: Dict[str, Any]):
    log_entry = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'event_type': event_type,
        'details': details
    }
    safety_audit_logs.append(log_entry)
    print(f"[SECURITY ALERT - {event_type}]: {details}")

# --- 5. Integrated Enterprise Safe Digital Twin ---
class EnterpriseSafeDigitalTwin:
    def __init__(self, name: str):
        self.name = name
        self.pii_detector = EnhancedPIIDetector()
        self.content_filter = DomainFilter()
        self.validator = OutputValidator()
        self.rate_limiter = RateLimiter(max_requests=3, window_seconds=5)

    def chat(self, user_input: str) -> str:
        # Check Rate Limit
        if not self.rate_limiter.is_allowed():
            log_security_event('RATE_LIMIT_EXCEEDED', {'user_input': user_input[:30] + '...'})
            return "Too many requests. Please slow down and try again later."

        # Step 1: Filter input
        filter_result = self.content_filter.filter(user_input)
        if not filter_result['is_safe']:
            log_security_event('INPUT_BLOCKED', {'violations': filter_result['sensitive_topics']})
            return "I cannot process that input. It violates our community guidelines."

        # Step 2: Check Input PII & Redact before sending to model
        pii_result = self.pii_detector.detect(user_input)
        clean_input = user_input
        if pii_result['has_pii']:
            log_security_event('PII_REDACTED', {'pii_types': pii_result['pii_types']})
            clean_input = self.pii_detector.redact(user_input)

        # Step 3: Mock response generation using cleaned/redacted input
        response = f"[Twin {self.name}] I processed your request: '{clean_input}'"

        # Step 4: Validate output
        validation = self.validator.validate(response)
        if not validation['is_valid']:
            log_security_event('OUTPUT_VALIDATION_FAILED', {'issues': validation['issues']})
            return "I apologize, but I generated a response that failed our safety checks."

        return response

# --- 6. Edge Cases & Robust Demonstration ---
twin = EnterpriseSafeDigitalTwin("SecuredTwin-v1")

print("=== Case 1: Normal input ===")
print(twin.chat("Hi there! Can we schedule a quick call tomorrow?"))

print("\n=== Case 2: PII Redaction (DOB & Address) ===")
print(twin.chat("My birthdate is 05-12-1990 and I live at 123 Maple Street."))

print("\n=== Case 3: Business/Topic Violations ===")
print(twin.chat("I want to leak_secrets about competitor_x."))

print("\n=== Case 4: Rate Limiter Trigger ===")
for i in range(5):
    print(f"Request {i+1}: {twin.chat('Ping')}")

print("\n=== Safety Logs Summary ===")
import json
print(json.dumps(safety_audit_logs, indent=2))

=== Case 1: Normal input ===
[Twin SecuredTwin-v1] I processed your request: 'Hi there! Can we schedule a quick call tomorrow?'

=== Case 2: PII Redaction (DOB & Address) ===
[SECURITY ALERT - PII_REDACTED]: {'pii_types': ['dob', 'address']}
[Twin SecuredTwin-v1] I processed your request: 'My birthdate is [REDACTED_DOB] and I live at [REDACTED_ADDRESS].'

=== Case 3: Business/Topic Violations ===
[SECURITY ALERT - INPUT_BLOCKED]: {'violations': ['competitor_x', 'leak_secrets']}
I cannot process that input. It violates our community guidelines.

=== Case 4: Rate Limiter Trigger ===
[SECURITY ALERT - RATE_LIMIT_EXCEEDED]: {'user_input': 'Ping...'}
Request 1: Too many requests. Please slow down and try again later.
[SECURITY ALERT - RATE_LIMIT_EXCEEDED]: {'user_input': 'Ping...'}
Request 2: Too many requests. Please slow down and try again later.
[SECURITY ALERT - RATE_LIMIT_EXCEEDED]: {'user_input': 'Ping...'}
Request 3: Too many requests. Please slow down and try again later.
[SECURITY 

## Summary

### What We Learned:
- Content filtering blocks inappropriate content
- PII detection protects user privacy
- Output validation ensures quality
- Multiple safety layers = robust system
- Always add disclaimers for sensitive topics

### Next: Week 3.4 - Red Teaming & Security